In [ ]:
# Vectorless RAG using PageIndex + Bedrock (Google Colab)

# #This notebook demonstrates a **reasoning-based, vectorless RAG pipeline**
# using **PageIndex** and **AWS Bedrock**.

# - No embeddings
# - No chunking
# - No vector database
# - Tree-based reasoning only

# PAGEINDEX API → grabs and indexes web or document content.

# Bedrock → takes that content and uses an LLM to summarize, answer questions, or generate insights.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("environment loaded")

assert os.getenv("PAGEINDEX_API_KEY") ,"Missing PAGEINDEX_API_KEY" #Make sure the environment variable PAGEINDEX_API_KEY exists

environment loaded


In [6]:
import json
import time
import asyncio
import pageindex
import boto3
from pageindex import PageIndexClient

In [ ]:
pageindex = PageIndexClient(api_key= os.getenv("PAGEINDEX_API_KEY")) #Creates a client object that knows how to talk to the PageIndex API

In [20]:
# Create Bedrock client (using system AWS credentials)
bedrock_client = boto3.client(
    service_name='bedrock-runtime',
    region_name='ap-southeast-2'  
    )

In [22]:
# Submit the PDF (local file path)
result = pageindex.submit_document(r"F:\Projects\Page_indexing project\DeepSeek-R1.pdf")
doc_id = result["doc_id"]  

print(" Save this doc_id:", doc_id)

 Save this doc_id: pi-cml0h8t8j00ki0gr1pybn7hh5


In [ ]:
#  Save this doc_id: pi-cml0h8t8j00ki0gr1pybn7hh5

In [25]:
def wait_until_ready(doc_id, timeout=300):
    start_time = time.time()
    while True:
        status = pageindex.get_document(doc_id)["status"]
        print("Status:", status)
        if status in ["ready", "completed"]:  # check for completed too
            print(" Document is ready!")
            break
        if time.time() - start_time > timeout:
            print(" Timeout reached. Stop waiting.")
            break
        time.sleep(5)

wait_until_ready(doc_id)


Status: completed
 Document is ready!


In [33]:
# Fetch the tree
tree = pageindex.get_tree(doc_id)

# Safely get the list of nodes
nodes = tree.get("result")

# Recursive function to print full hierarchy
def print_tree(nodes, depth=0):
    for node in nodes:
        # Print the current node with indentation
        print("  " * depth + f"- [{node.get('node_id')}] {node.get('title')}")
        
        # If this node has children, recursively print them
        children = node.get("children")
        if children:
            print_tree(children, depth + 1)

# Print the full document tree
print_tree(nodes)

- [0000] Abstract
- [0001] 1 Introduction
- [0002] 2 DeepSeek-R1-Zero
- [0003] 3. DeepSeek-R1
- [0004] 4 Experiment
- [0005] 5 Ethics and Safety Statement
- [0006] 6 Conclusion, Limitation, and Future Work
- [0007] 7 Author List
- [0008] Appendix A Background
- [0009] B. Training Details
- [0017] C. Self-Evolution of DeepSeek-R1-Zero
- [0018] D. Evaluation of DeepSeek-R1
- [0026] E. More Analysis
- [0027] F. DeepSeek-R1 Distillation
- [0028] G. Discussion
- [0029] Appendix H Related Work
- [0030] Appendix I Open Weights, Code, and Data
- [0031] J. Evaluation Prompts and Settings


In [38]:
def strip_text(nodes):
    return [
        {
            "node_id": n["node_id"],
            "title": n.get("title"),
            "summary": n.get("summary"),
            "children": strip_text(n.get("children",[]))
        }
        for n in nodes
    ]

# Safely get nodes from the tree
nodes = tree.get("result",[])
reasoning_tree = strip_text(nodes)

In [39]:
QUESTION = "What are the conclusions of this paper?"

In [ ]:
REASONING_PROMPT = f"""
You are given a document structure as a tree.

Each node has:
- node_id
- title
- summary
- children

Question:
{QUESTION}

Task:
Select the most relevant node IDs.
Return JSON ONLY:

{{
  "reasoning": "...",
  "selected_node_ids": ["node_id"]
}}

Tree:
{json.dumps(reasoning_tree, indent=2)}
"""

In [42]:
print(REASONING_PROMPT)


You are given a document structure as a tree.

Each node has:
- node_id
- title
- summary

Question:
What are the conclusions of this paper?

Task:
Select the most relevant node IDs.
Return JSON ONLY:

{
  "reasoning": "...",
  "selected_node_ids": ["node_id"]
}

Tree:
[
  {
    "node_id": "0000",
    "title": "Abstract",
    "summary": null,
    "children": []
  },
  {
    "node_id": "0001",
    "title": "1 Introduction",
    "summary": null,
    "children": []
  },
  {
    "node_id": "0002",
    "title": "2 DeepSeek-R1-Zero",
    "summary": null,
    "children": []
  },
  {
    "node_id": "0003",
    "title": "3. DeepSeek-R1",
    "summary": null,
    "children": []
  },
  {
    "node_id": "0004",
    "title": "4 Experiment",
    "summary": null,
    "children": []
  },
  {
    "node_id": "0005",
    "title": "5 Ethics and Safety Statement",
    "summary": null,
    "children": []
  },
  {
    "node_id": "0006",
    "title": "6 Conclusion, Limitation, and Future Work",
    "summary"

In [ ]:
async def reason_over_tree(reasoning_prompt: str):
    """
    Invoke Anthropic Claude Messages API on Amazon Bedrock.
    Returns parsed JSON from Claude.
    """

    request_body = {
        "anthropic_version": "bedrock-2023-05-31", 
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": reasoning_prompt}
                ]
            }
        ],
        "max_tokens": 1024,
        "temperature": 0.0
    }

    response = bedrock_client.invoke_model(
        modelId="apac.anthropic.claude-3-5-sonnet-20240620-v1:0",
        body=json.dumps(request_body),
        accept="application/json",
        contentType="application/json"
    )

    body = json.loads(response["body"].read())

    reply_text = body["content"][0]["text"]

    # Parse Claude's JSON-only response
    try:
        return json.loads(reply_text)
    except json.JSONDecodeError:
        return {
            "error": "Claude did not return valid JSON",
            "raw_output": reply_text
        }
reasoning_output = await reason_over_tree(REASONING_PROMPT)
print(json.dumps(reasoning_output, indent=2))


{
  "reasoning": "The conclusions of a paper are typically found in the conclusion section. In this document structure, the most relevant section for conclusions would be '6 Conclusion, Limitation, and Future Work' with node_id '0006'. This section is likely to summarize the main findings, discuss limitations, and suggest future research directions.",
  "selected_node_ids": [
    "0006"
  ]
}


In [71]:
# Fetch the tree
tree = pageindex.get_tree(doc_id)

# Safely get the list of nodes
nodes = tree.get("result", [])

#  Flatten the tree into a dict for easy lookup
def flatten_tree(nodes, flat=None):
    if flat is None:
        flat = {}
    for node in nodes:
        flat[node["node_id"]] = node
        children = node.get("children", [])
        if children:
            flatten_tree(children, flat)
    return flat

flat_nodes = flatten_tree(nodes) 


In [74]:
#  Collect text from selected_node_ids
texts = []
selected_node_ids = reasoning_output.get("selected_node_ids")

for node_id in selected_node_ids:
    node = flat_nodes.get(str(node_id))  # make sure node_id is a string
    if node:
        texts.append(node.get("text"))  # replace "text" if your text field has a different name

# Combine all text into one string
context = "\n\n".join(texts)

In [75]:
#  Optional: check the first few characters
print(context[:500])

# 6 Conclusion, Limitation, and Future Work

We present DeepSeek-R1-Zero and DeepSeek-R1, which rely on large-scale RL to incentivize model reasoning behaviors. Our results demonstrate that pre-trained checkpoints inherently possess substantial potential for complex reasoning tasks. We believe that the key to unlocking this potential lies not in large-scale human annotation but in the provision of hard reasoning questions, a reliable verifier, and sufficient computational resources for reinforce


In [76]:
ANSWER_PROMPT = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{QUESTION}
"""

In [79]:
async def generate_answer(prompt: str):
    """
    Invoke Anthropic Claude Messages API on Amazon Bedrock.
    Returns the text response from Claude.
    """
    
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt}
                ]
            }
        ],
        "max_tokens": 500,
        "temperature": 0.0
    }

    response = bedrock_client.invoke_model(
        modelId="apac.anthropic.claude-3-5-sonnet-20240620-v1:0",
        body=json.dumps(request_body),
        accept="application/json",
        contentType="application/json"
    )

    body = json.loads(response["body"].read())
    reply_text = body["content"][0]["text"]

    # Optionally, try parsing as JSON first
    try:
        return json.loads(reply_text)
    except json.JSONDecodeError:
        return reply_text


In [80]:
# Generate final answer
final_answer = await generate_answer(ANSWER_PROMPT)
print(final_answer)

Based on the provided context, the main conclusions of this paper are:

1. DeepSeek-R1-Zero and DeepSeek-R1 models were developed using large-scale reinforcement learning to enhance model reasoning behaviors.

2. Pre-trained checkpoints have significant potential for complex reasoning tasks, which can be unlocked through hard reasoning questions, reliable verifiers, and sufficient computational resources for reinforcement learning.

3. Sophisticated reasoning behaviors like self-verification and reflection emerged organically during the reinforcement learning process.

4. The models achieved frontier results on reasoning benchmarks, demonstrating the effectiveness of the approach.

5. Despite the achievements, the models still face several limitations, including issues with structural output, tool use, token efficiency, language mixing, and performance on software engineering tasks.

6. The pure reinforcement learning methodology presents challenges, particularly in terms of reward hac